In [ ]:
!pip install -q deep-translator sentence-transformers

In [71]:
import joblib
import numpy as np
import pandas as pd

model_dir = "/content/drive/MyDrive/medical_triage_project/models/"

baseline_model = joblib.load(model_dir + "classifier.pkl")
retrieval_embeddings = np.load(model_dir + "retrieval_embeddings.npy")
retrieval_df = pd.read_csv(model_dir + "retrieval_data.csv")

print("✅ Model ve veriler yüklendi")

✅ Model ve veriler yüklendi


In [2]:
!pip install -q datasets huggingface_hub pandas

In [3]:
from datasets import load_dataset

ds = load_dataset("gretelai/symptom_to_diagnosis")
ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/853 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/212 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output_text', 'input_text'],
        num_rows: 853
    })
    test: Dataset({
        features: ['output_text', 'input_text'],
        num_rows: 212
    })
})

In [4]:
train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

print(train_df.head())
print(test_df.head())
print(train_df.columns)

               output_text                                         input_text
0     cervical spondylosis  I've been having a lot of pain in my neck and ...
1                 impetigo  I have a rash on my face that is getting worse...
2  urinary tract infection  I have been urinating blood. I sometimes feel ...
3                arthritis  I have been having trouble with my muscles and...
4                   dengue  I have been feeling really sick. My body hurts...
            output_text                                         input_text
0  peptic ulcer disease  I have a burning sensation in my stomach that ...
1  peptic ulcer disease  I have a hard time swallowing and I feel like ...
2         drug reaction  I've been having headaches and migraines, and ...
3             pneumonia  I'm sweating a lot and can't catch my breath. ...
4      fungal infection  I've been scratching myself a lot lately, and ...
Index(['output_text', 'input_text'], dtype='object')


In [5]:
import os

save_path = "/content/drive/MyDrive/medical_triage_project/data/raw/"

os.makedirs(save_path, exist_ok=True)

In [6]:

train_df.to_csv(save_path + "gretelai_train.csv", index=False)
test_df.to_csv(save_path + "gretelai_test.csv", index=False)

print("Kaydedildi.")

Kaydedildi.


In [7]:
import  pandas as pd
df = pd.read_csv("/content/drive/MyDrive/medical_triage_project/data/raw/gretelai_train.csv")

print(df.head())
print(df.columns)
print(df.shape)

               output_text                                         input_text
0     cervical spondylosis  I've been having a lot of pain in my neck and ...
1                 impetigo  I have a rash on my face that is getting worse...
2  urinary tract infection  I have been urinating blood. I sometimes feel ...
3                arthritis  I have been having trouble with my muscles and...
4                   dengue  I have been feeling really sick. My body hurts...
Index(['output_text', 'input_text'], dtype='object')
(853, 2)


In [8]:
df1= pd.read_csv("Final_Augmented_dataset_Diseases_and_Symptoms.csv")

In [9]:
df2= pd.read_csv("Symptom2Disease.csv")

In [10]:
print(df1.head())

         diseases  anxiety and nervousness  depression  shortness of breath  \
0  panic disorder                        1           0                    1   
1  panic disorder                        0           0                    1   
2  panic disorder                        1           1                    1   
3  panic disorder                        1           0                    0   
4  panic disorder                        1           1                    0   

   depressive or psychotic symptoms  sharp chest pain  dizziness  insomnia  \
0                                 1                 0          0         0   
1                                 1                 0          1         1   
2                                 1                 0          1         1   
3                                 1                 0          1         1   
4                                 0                 0          0         1   

   abnormal involuntary movements  chest tightness  ... 

In [11]:
print(df2.head())

   Unnamed: 0      label                                               text
0           0  Psoriasis  I have been experiencing a skin rash on my arm...
1           1  Psoriasis  My skin has been peeling, especially on my kne...
2           2  Psoriasis  I have been experiencing joint pain in my fing...
3           3  Psoriasis  There is a silver like dusting on my skin, esp...
4           4  Psoriasis  My nails have small dents or pits in them, and...


In [12]:
symptom2_std = pd.DataFrame({
    "text": df2["text"].astype(str).str.strip(),
    "label": df2["label"].astype(str).str.strip().str.lower(),
    "source": "symptom2disease",
    "style": "natural",
    "language": "en"
})

In [13]:
gretelai_std = pd.DataFrame({
    "text": train_df["input_text"].astype(str).str.strip(),
    "label": train_df["output_text"].astype(str).str.strip().str.lower(),
    "source": "gretelai",
    "style": "natural",
    "language": "en"
})

In [14]:
classifier_df = pd.concat([symptom2_std, gretelai_std], ignore_index=True)

In [15]:
symptom_cols = [col for col in df1.columns if col != "diseases"]

print(len(symptom_cols))  # ~377 olması lazım

377


In [16]:
def row_to_text(row, symptom_cols):
    active = []
    for col in symptom_cols:
        if row[col] == 1 or row[col] == 1.0:
            clean = col.strip().lower()
            active.append(clean)
    return "; ".join(active)

In [17]:
structured_std = pd.DataFrame({
    "text": df1.apply(lambda row: row_to_text(row, symptom_cols), axis=1),
    "label": df1["diseases"].astype(str).str.strip().str.lower(),
    "source": "structured_dataset",
    "style": "structured",
    "language": "en"
})

In [18]:
print(structured_std.head())
print(structured_std.shape)

                                                text           label  \
0  anxiety and nervousness; shortness of breath; ...  panic disorder   
1  shortness of breath; depressive or psychotic s...  panic disorder   
2  anxiety and nervousness; depression; shortness...  panic disorder   
3  anxiety and nervousness; depressive or psychot...  panic disorder   
4  anxiety and nervousness; depression; insomnia;...  panic disorder   

               source       style language  
0  structured_dataset  structured       en  
1  structured_dataset  structured       en  
2  structured_dataset  structured       en  
3  structured_dataset  structured       en  
4  structured_dataset  structured       en  
(36654, 5)


In [19]:
structured_std = structured_std[structured_std["text"] != ""]
structured_std = structured_std.reset_index(drop=True)

In [20]:
save_path = "/content/drive/MyDrive/medical_triage_project/data/processed/"
import os
os.makedirs(save_path, exist_ok=True)

structured_std.to_csv(save_path + "retrieval_dataset.csv", index=False)

print("structured_std kaydedildi")

structured_std kaydedildi


In [21]:
print(classifier_df.head())
print(classifier_df.shape)
print(classifier_df["source"].value_counts())
print(classifier_df["label"].nunique())
print(classifier_df.isnull().sum())

                                                text      label  \
0  I have been experiencing a skin rash on my arm...  psoriasis   
1  My skin has been peeling, especially on my kne...  psoriasis   
2  I have been experiencing joint pain in my fing...  psoriasis   
3  There is a silver like dusting on my skin, esp...  psoriasis   
4  My nails have small dents or pits in them, and...  psoriasis   

            source    style language  
0  symptom2disease  natural       en  
1  symptom2disease  natural       en  
2  symptom2disease  natural       en  
3  symptom2disease  natural       en  
4  symptom2disease  natural       en  
(2053, 5)
source
symptom2disease    1200
gretelai            853
Name: count, dtype: int64
24
text        0
label       0
source      0
style       0
language    0
dtype: int64


In [22]:
print(structured_std.head())
print(structured_std.shape)
print(structured_std["label"].nunique())
print(structured_std.isnull().sum())

                                                text           label  \
0  anxiety and nervousness; shortness of breath; ...  panic disorder   
1  shortness of breath; depressive or psychotic s...  panic disorder   
2  anxiety and nervousness; depression; shortness...  panic disorder   
3  anxiety and nervousness; depressive or psychot...  panic disorder   
4  anxiety and nervousness; depression; insomnia;...  panic disorder   

               source       style language  
0  structured_dataset  structured       en  
1  structured_dataset  structured       en  
2  structured_dataset  structured       en  
3  structured_dataset  structured       en  
4  structured_dataset  structured       en  
(36654, 5)
120
text        0
label       0
source      0
style       0
language    0
dtype: int64


In [23]:
classifier_df = classifier_df.drop_duplicates().reset_index(drop=True)
structured_std = structured_std.drop_duplicates().reset_index(drop=True)

In [24]:
print("classifier:", classifier_df.shape)
print("retrieval:", structured_std.shape)

classifier: (2002, 5)
retrieval: (27353, 5)


In [25]:
classifier_df["label"] = classifier_df["label"].astype(str).str.strip().str.lower()
structured_std["label"] = structured_std["label"].astype(str).str.strip().str.lower()

classifier_df["text"] = classifier_df["text"].astype(str).str.strip()
structured_std["text"] = structured_std["text"].astype(str).str.strip()

In [26]:
classifier_labels = set(classifier_df["label"].unique())
retrieval_labels = set(structured_std["label"].unique())

print("Classifier label sayısı:", len(classifier_labels))
print("Retrieval label sayısı:", len(retrieval_labels))
print("Ortak label sayısı:", len(classifier_labels.intersection(retrieval_labels)))

Classifier label sayısı: 24
Retrieval label sayısı: 120
Ortak label sayısı: 0


In [27]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    classifier_df,
    test_size=0.2,
    stratify=classifier_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (1601, 5)
val: (200, 5)
test: (201, 5)


In [28]:
save_path = "/content/drive/MyDrive/medical_triage_project/data/processed/"

train_df.to_csv(save_path + "classifier_train.csv", index=False)
val_df.to_csv(save_path + "classifier_val.csv", index=False)
test_df.to_csv(save_path + "classifier_test.csv", index=False)

print("classifier splitleri kaydedildi")

classifier splitleri kaydedildi


In [29]:
#split soyalarını oku

In [30]:
import pandas as pd

save_path = "/content/drive/MyDrive/medical_triage_project/data/processed/"

train_df = pd.read_csv(save_path + "classifier_train.csv")
val_df   = pd.read_csv(save_path + "classifier_val.csv")
test_df  = pd.read_csv(save_path + "classifier_test.csv")

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

display(train_df.head())

train: (1601, 5)
val: (200, 5)
test: (201, 5)


,text,label,source,style,language
0,"I have trouble breathing, especially when I'm ...",diabetes,gretelai,natural,en
1,I've been finding it challenging to use the re...,dimorphic hemorrhoids,symptom2disease,natural,en
2,I've noticed that my skin is more sensitive no...,psoriasis,gretelai,natural,en
3,The nausea I have been feeling is accompanied ...,dengue,symptom2disease,natural,en
4,I'm having a lot of problems breathing. I'm no...,pneumonia,symptom2disease,natural,en


In [31]:
#x ve y ayır

In [32]:
X_train = train_df["text"].astype(str)
y_train = train_df["label"].astype(str)

X_val = val_df["text"].astype(str)
y_val = val_df["label"].astype(str)

X_test = test_df["text"].astype(str)
y_test = test_df["label"].astype(str)

In [33]:
#baseline kur
#Bu bir text classifier. Metni önce TF-IDF ile vektöre çeviriyor, sonra sınıflandırma yapıyor.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=20000
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

baseline_model.fit(X_train, y_train)
print("Model eğitildi.")

Model eğitildi.


In [34]:
from sklearn.metrics import accuracy_score, classification_report

val_pred = baseline_model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, val_pred))
print(classification_report(y_val, val_pred, zero_division=0))

Validation Accuracy: 0.975
                                 precision    recall  f1-score   support

                           acne       1.00      1.00      1.00         5
                        allergy       1.00      0.89      0.94         9
                      arthritis       1.00      1.00      1.00         9
               bronchial asthma       0.90      1.00      0.95         9
           cervical spondylosis       1.00      1.00      1.00         9
                    chicken pox       0.90      1.00      0.95         9
                    common cold       1.00      1.00      1.00         9
                         dengue       1.00      0.89      0.94         9
                       diabetes       1.00      1.00      1.00         9
          dimorphic hemorrhoids       1.00      1.00      1.00         4
                  drug reaction       1.00      1.00      1.00         9
               fungal infection       1.00      1.00      1.00         9
gastroesophageal reflux

In [35]:
test_pred = baseline_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, test_pred))
print(classification_report(y_test, test_pred, zero_division=0))

Test Accuracy: 0.9900497512437811
                                 precision    recall  f1-score   support

                           acne       1.00      1.00      1.00         4
                        allergy       1.00      1.00      1.00         9
                      arthritis       1.00      1.00      1.00         8
               bronchial asthma       1.00      1.00      1.00         9
           cervical spondylosis       1.00      1.00      1.00         9
                    chicken pox       1.00      1.00      1.00         9
                    common cold       1.00      1.00      1.00         9
                         dengue       1.00      0.89      0.94         9
                       diabetes       1.00      1.00      1.00         9
          dimorphic hemorrhoids       1.00      1.00      1.00         4
                  drug reaction       1.00      0.89      0.94         9
               fungal infection       1.00      1.00      1.00         9
gastroesophageal

In [36]:
import numpy as np

classes = baseline_model.named_steps["clf"].classes_
probs = baseline_model.predict_proba(X_test)

def get_top_k_predictions(prob_row, classes, k=3):
    idx = np.argsort(prob_row)[-k:][::-1]
    return [(classes[i], float(prob_row[i])) for i in idx]

for i in range(5):
    print("TEXT:", X_test.iloc[i])
    print("TRUE LABEL:", y_test.iloc[i])
    print("TOP-3:", get_top_k_predictions(probs[i], classes, k=3))
    print("-" * 100)

TEXT: I have been getting blood in my pee. Sometimes I get nauseous while peeing. This often almost coincides with me having a high temperature
TRUE LABEL: urinary tract infection
TOP-3: [('urinary tract infection', 0.3633133565708372), ('drug reaction', 0.05519253744316418), ('peptic ulcer disease', 0.05254168533351684)]
----------------------------------------------------------------------------------------------------
TEXT: A rash that appears to be developing throughout my skin has been accompanying my recent bouts of intense itching and discomfort. On my skin, I also have some dischromic spots and little lumps that seem to be appearing everywhere.
TRUE LABEL: fungal infection
TOP-3: [('fungal infection', 0.24122266514366164), ('psoriasis', 0.07851313180759882), ('diabetes', 0.06372199693916036)]
----------------------------------------------------------------------------------------------------
TEXT: I've been feeling extremely scratchy, sick, and worn out. In addition, I've lost 

In [37]:
import os
import joblib

model_dir = "/content/drive/MyDrive/medical_triage_project/models/"
os.makedirs(model_dir, exist_ok=True)

joblib.dump(baseline_model, model_dir + "baseline_tfidf_logreg.pkl")
print("Model kaydedildi.")

Model kaydedildi.


In [38]:
sample_text = "i feel like vomiting and stomach hurts"

pred = baseline_model.predict([sample_text])[0]
pred_probs = baseline_model.predict_proba([sample_text])[0]

print("Prediction:", pred)
print("Top-3:", get_top_k_predictions(pred_probs, classes, k=3))

Prediction: typhoid
Top-3: [('typhoid', 0.18389333321570014), ('peptic ulcer disease', 0.0806865552559329), ('urinary tract infection', 0.0695087555755071)]


In [39]:
!pip install -q sentence-transformers

In [40]:
import pandas as pd

processed_path = "/content/drive/MyDrive/medical_triage_project/data/processed/"

retrieval_df = pd.read_csv(processed_path + "retrieval_dataset.csv")

print(retrieval_df.shape)
display(retrieval_df.head())

(36654, 5)


,text,label,source,style,language
0,anxiety and nervousness; shortness of breath; ...,panic disorder,structured_dataset,structured,en
1,shortness of breath; depressive or psychotic s...,panic disorder,structured_dataset,structured,en
2,anxiety and nervousness; depression; shortness...,panic disorder,structured_dataset,structured,en
3,anxiety and nervousness; depressive or psychot...,panic disorder,structured_dataset,structured,en
4,anxiety and nervousness; depression; insomnia;...,panic disorder,structured_dataset,structured,en


In [41]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding modeli yüklendi.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding modeli yüklendi.


In [42]:
retrieval_texts = retrieval_df["text"].astype(str).tolist()

retrieval_embeddings = embed_model.encode(
    retrieval_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(retrieval_embeddings.shape)

Batches:   0%|          | 0/1146 [00:00<?, ?it/s]

(36654, 384)


In [43]:
import numpy as np
import os

model_dir = "/content/drive/MyDrive/medical_triage_project/models/"
os.makedirs(model_dir, exist_ok=True)

np.save(model_dir + "retrieval_embeddings.npy", retrieval_embeddings)
print("Retrieval embeddingleri kaydedildi.")

Retrieval embeddingleri kaydedildi.


In [44]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_similar_cases(query_text, retrieval_df, retrieval_embeddings, embed_model, top_k=5):
    query_embedding = embed_model.encode([query_text], convert_to_numpy=True)

    sims = cosine_similarity(query_embedding, retrieval_embeddings)[0]
    top_idx = np.argsort(sims)[-top_k:][::-1]

    results = retrieval_df.iloc[top_idx].copy()
    results["similarity"] = sims[top_idx]

    return results[["text", "label", "similarity"]].reset_index(drop=True)

In [45]:
query = "I have had cough, sore throat and fever for 3 days"

results = retrieve_similar_cases(
    query_text=query,
    retrieval_df=retrieval_df,
    retrieval_embeddings=retrieval_embeddings,
    embed_model=embed_model,
    top_k=5
)

display(results)

,text,label,similarity
0,sore throat; cough; fever,chronic sinusitis,0.793798
1,sore throat; cough; fever,acute sinusitis,0.793798
2,sore throat; fever,chronic obstructive pulmonary disease (copd),0.750280
3,sore throat; fever,acute sinusitis,0.750280
4,sore throat; wheezing; fever,chronic obstructive pulmonary disease (copd),0.743676


In [46]:
import numpy as np

classes = baseline_model.named_steps["clf"].classes_

def get_top_k_predictions(prob_row, classes, k=3):
    idx = np.argsort(prob_row)[-k:][::-1]
    return [(classes[i], float(prob_row[i])) for i in idx]

def predict_with_retrieval(user_text, baseline_model, classes, retrieval_df, retrieval_embeddings, embed_model, top_k_cls=3, top_k_ret=5):
    pred_probs = baseline_model.predict_proba([user_text])[0]
    top_preds = get_top_k_predictions(pred_probs, classes, k=top_k_cls)

    retrieved = retrieve_similar_cases(
        query_text=user_text,
        retrieval_df=retrieval_df,
        retrieval_embeddings=retrieval_embeddings,
        embed_model=embed_model,
        top_k=top_k_ret
    )

    return {
        "input_text": user_text,
        "top_predictions": top_preds,
        "retrieved_cases": retrieved
    }

In [47]:
user_text = "I have had cough, sore throat and fever for 3 days"

result = predict_with_retrieval(
    user_text=user_text,
    baseline_model=baseline_model,
    classes=classes,
    retrieval_df=retrieval_df,
    retrieval_embeddings=retrieval_embeddings,
    embed_model=embed_model,
    top_k_cls=3,
    top_k_ret=5
)

print("INPUT:")
print(result["input_text"])

print("\nTOP-3 PREDICTIONS:")
for label, score in result["top_predictions"]:
    print(f"{label}: {score:.4f}")

print("\nRETRIEVED CASES:")
display(result["retrieved_cases"])

INPUT:
I have had cough, sore throat and fever for 3 days

TOP-3 PREDICTIONS:
allergy: 0.1483
common cold: 0.0755
bronchial asthma: 0.0702

RETRIEVED CASES:


,text,label,similarity
0,sore throat; cough; fever,chronic sinusitis,0.793798
1,sore throat; cough; fever,acute sinusitis,0.793798
2,sore throat; fever,chronic obstructive pulmonary disease (copd),0.750280
3,sore throat; fever,acute sinusitis,0.750280
4,sore throat; wheezing; fever,chronic obstructive pulmonary disease (copd),0.743676


In [48]:
retrieved_labels = result["retrieved_cases"]["label"].value_counts()
print(retrieved_labels)

label
acute sinusitis                                 2
chronic obstructive pulmonary disease (copd)    2
chronic sinusitis                               1
Name: count, dtype: int64


In [49]:
# =========================
# FINAL INFERENCE BLOĞU
# classifier + retrieval + özet + final yorum
# =========================

import numpy as np
from collections import Counter

# 1) Top-k prediction fonksiyonu
def get_top_k_predictions(prob_row, classes, k=3):
    idx = np.argsort(prob_row)[-k:][::-1]
    return [(classes[i], float(prob_row[i])) for i in idx]

# 2) Retrieval fonksiyonu
def retrieve_similar_cases(query_text, retrieval_df, retrieval_embeddings, embed_model, top_k=5):
    query_embedding = embed_model.encode([query_text], convert_to_numpy=True)
    sims = cosine_similarity(query_embedding, retrieval_embeddings)[0]
    top_idx = np.argsort(sims)[-top_k:][::-1]

    results = retrieval_df.iloc[top_idx].copy()
    results["similarity"] = sims[top_idx]

    return results[["text", "label", "similarity"]].reset_index(drop=True)

# 3) Retrieval label özeti
def summarize_retrieved_labels(retrieved_cases):
    label_counts = retrieved_cases["label"].value_counts().to_dict()
    return label_counts

# 4) Final yorum üret
def build_final_comment(top_predictions, retrieved_label_summary):
    top1_label = top_predictions[0][0]
    top1_score = top_predictions[0][1]

    if len(retrieved_label_summary) == 0:
        return "No similar retrieved cases were found."

    retrieved_top_label = max(retrieved_label_summary, key=retrieved_label_summary.get)
    retrieved_top_count = retrieved_label_summary[retrieved_top_label]

    # classifier top-1 retrieval'da da destekleniyorsa
    if top1_label in retrieved_label_summary:
        return (
            f"Classifier top prediction is '{top1_label}' "
            f"(score={top1_score:.3f}) and retrieval also supports this label."
        )

    # classifier top-1 retrieval ile uyuşmuyorsa
    return (
        f"Classifier top prediction is '{top1_label}' (score={top1_score:.3f}), "
        f"but retrieved cases are more consistent with '{retrieved_top_label}' "
        f"({retrieved_top_count} similar case(s))."
    )

# 5) Ana fonksiyon
def full_predict(user_text, baseline_model, classes, retrieval_df, retrieval_embeddings, embed_model,
                 top_k_cls=3, top_k_ret=5):

    # classifier
    pred_probs = baseline_model.predict_proba([user_text])[0]
    top_predictions = get_top_k_predictions(pred_probs, classes, k=top_k_cls)

    # retrieval
    retrieved_cases = retrieve_similar_cases(
        query_text=user_text,
        retrieval_df=retrieval_df,
        retrieval_embeddings=retrieval_embeddings,
        embed_model=embed_model,
        top_k=top_k_ret
    )

    # retrieval özeti
    retrieved_label_summary = summarize_retrieved_labels(retrieved_cases)

    # basit semptom özeti
    summary = f"Patient-reported complaint: {user_text}"

    # final yorum
    final_comment = build_final_comment(top_predictions, retrieved_label_summary)

    return {
        "input_text": user_text,
        "summary": summary,
        "top_predictions": top_predictions,
        "retrieved_cases": retrieved_cases,
        "retrieved_label_summary": retrieved_label_summary,
        "final_comment": final_comment
    }

# 6) Sonucu güzel bastır
def print_full_result(result):
    print("INPUT:")
    print(result["input_text"])

    print("\nSUMMARY:")
    print(result["summary"])

    print("\nTOP PREDICTIONS:")
    for i, (label, score) in enumerate(result["top_predictions"], start=1):
        print(f"{i}. {label} -> {score:.4f}")

    print("\nRETRIEVED LABEL SUMMARY:")
    for label, count in result["retrieved_label_summary"].items():
        print(f"- {label}: {count}")

    print("\nFINAL COMMENT:")
    print(result["final_comment"])

    print("\nRETRIEVED CASES:")
    display(result["retrieved_cases"])

In [50]:
user_text = "I have had cough, sore throat and fever for 3 days"

result = full_predict(
    user_text=user_text,
    baseline_model=baseline_model,
    classes=classes,
    retrieval_df=retrieval_df,
    retrieval_embeddings=retrieval_embeddings,
    embed_model=embed_model,
    top_k_cls=3,
    top_k_ret=5
)

print_full_result(result)

INPUT:
I have had cough, sore throat and fever for 3 days

SUMMARY:
Patient-reported complaint: I have had cough, sore throat and fever for 3 days

TOP PREDICTIONS:
1. allergy -> 0.1483
2. common cold -> 0.0755
3. bronchial asthma -> 0.0702

RETRIEVED LABEL SUMMARY:
- acute sinusitis: 2
- chronic obstructive pulmonary disease (copd): 2
- chronic sinusitis: 1

FINAL COMMENT:
Classifier top prediction is 'allergy' (score=0.148), but retrieved cases are more consistent with 'acute sinusitis' (2 similar case(s)).

RETRIEVED CASES:


,text,label,similarity
0,sore throat; cough; fever,chronic sinusitis,0.793798
1,sore throat; cough; fever,acute sinusitis,0.793798
2,sore throat; fever,chronic obstructive pulmonary disease (copd),0.750280
3,sore throat; fever,acute sinusitis,0.750280
4,sore throat; wheezing; fever,chronic obstructive pulmonary disease (copd),0.743676


In [51]:
!pip install deep-translator
from deep_translator import GoogleTranslator

def translate_to_en(text):
    return GoogleTranslator(source='auto', target='en').translate(text)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.8 MB/s eta 0:00:00


In [52]:
test_text_tr = "3 gündür öksürüyorum, boğazım ağrıyor ve ateşim var"
translated = translate_to_en(test_text_tr)

print("ORIGINAL:", test_text_tr)
print("TRANSLATED:", translated)

ORIGINAL: 3 gündür öksürüyorum, boğazım ağrıyor ve ateşim var
TRANSLATED: I have been coughing, my throat hurts and I have a fever for 3 days.


In [53]:
test_text_en = "I have had cough, sore throat and fever for 3 days"
translated_en = translate_to_en(test_text_en)

print("ORIGINAL:", test_text_en)
print("TRANSLATED:", translated_en)

ORIGINAL: I have had cough, sore throat and fever for 3 days
TRANSLATED: I have had cough, sore throat and fever for 3 days


In [54]:
from deep_translator import GoogleTranslator

def translate_to_tr(text):
    try:
        return GoogleTranslator(source='auto', target='tr').translate(text)
    except Exception as e:
        print("Translation error:", e)
        return text

In [55]:
def translate_top_predictions(top_predictions):
    translated = []
    for label, score in top_predictions:
        label_tr = translate_to_tr(label)
        translated.append((label_tr, score))
    return translated

In [56]:
def translate_retrieved_cases(retrieved_cases):
    translated_df = retrieved_cases.copy()

    # label çevir
    translated_df["label_tr"] = translated_df["label"].apply(translate_to_tr)

    # text çevirisi istersen bunu da aç
    translated_df["text_tr"] = translated_df["text"].apply(translate_to_tr)

    return translated_df

In [57]:
def localize_result_to_tr(result):
    localized = {}

    localized["input_text_original"] = result["input_text"]
    localized["input_text_tr"] = translate_to_tr(result["input_text"])
    localized["summary_tr"] = translate_to_tr(result["summary"])

    localized["top_predictions_tr"] = translate_top_predictions(result["top_predictions"])

    localized["pattern_note_tr"] = translate_to_tr(result["pattern_note"])
    localized["final_comment_tr"] = translate_to_tr(result["final_comment"])

    localized["retrieved_cases_tr"] = translate_retrieved_cases(result["retrieved_cases"])

    return localized

In [58]:
def print_localized_result_tr(localized_result):
    print("KULLANICI GİRİŞİ:")
    print(localized_result["input_text_tr"])

    print("\nÖZET:")
    print(localized_result["summary_tr"])

    print("\nOLASI DURUMLAR:")
    for i, (label_tr, score) in enumerate(localized_result["top_predictions_tr"], start=1):
        print(f"{i}. {label_tr} -> {score:.4f}")

    print("\nÖRÜNTÜ NOTU:")
    print(localized_result["pattern_note_tr"])

    print("\nSON YORUM:")
    print(localized_result["final_comment_tr"])

    print("\nUYARI:")
    print("Bu çıktı yalnızca semptom temelli karar destek amaçlıdır. Tıbbi tanı değildir.")

    print("\nBENZER VAKALAR (TR):")
    display(localized_result["retrieved_cases_tr"][["text_tr", "label_tr", "similarity"]])

In [59]:
def run_full_system_tr(user_text):
    # 1) kullanıcı girişini İngilizceye çevir
    translated_input = translate_to_en(user_text)

    # 2) İngilizce model pipeline'ını çalıştır
    result_en = predict_with_retrieval(
        user_text=translated_input,
        baseline_model=baseline_model,
        classes=classes,
        retrieval_df=retrieval_df,
        retrieval_embeddings=retrieval_embeddings,
        embed_model=embed_model,
        top_k_cls=3,
        top_k_ret=5
    )

    # 3) predict_with_retrieval sadece bu alanları dönüyor olabilir
    # eksik alanları ekleyelim
    if "summary" not in result_en:
        result_en["summary"] = f"Patient-reported complaint: {translated_input}"

    if "pattern_note" not in result_en:
        result_en["pattern_note"] = "Retrieved cases suggest a symptom similarity pattern."

    if "final_comment" not in result_en:
        result_en["final_comment"] = (
            "These outputs represent possible conditions based on symptom similarity "
            "and are not a medical diagnosis."
        )

    # 4) retrieval label summary yoksa üret
    if "retrieved_label_summary" not in result_en:
        result_en["retrieved_label_summary"] = result_en["retrieved_cases"]["label"].value_counts().to_dict()

    # 5) kullanıcı için Türkçeleştir
    result_tr = localize_result_to_tr(result_en)

    print("MODELİN GÖRDÜĞÜ İNGİLİZCE INPUT:")
    print(translated_input)

    print("\n--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---")
    print_localized_result_tr(result_tr)

    return result_en, result_tr

In [60]:
result_en, result_tr = run_full_system_tr("3 gündür öksürüyorum, boğazım ağrıyor ve ateşim var")

MODELİN GÖRDÜĞÜ İNGİLİZCE INPUT:
I have been coughing, my throat hurts and I have a fever for 3 days.

--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---
KULLANICI GİRİŞİ:
3 gündür öksürüyorum, boğazım ağrıyor ve ateşim var.

ÖZET:
Hastanın bildirdiği şikayet: 3 gündür öksürüyorum, boğazım ağrıyor ve ateşim var.

OLASI DURUMLAR:
1. nezle, soğuk algınlığı -> 0.1521
2. bronşiyal astım -> 0.0954
3. alerji -> 0.0606

ÖRÜNTÜ NOTU:
Alınan vakalar bir semptom benzerliği modelini akla getiriyor.

SON YORUM:
Bu çıktılar semptom benzerliğine dayalı olası koşulları temsil eder ve tıbbi bir teşhis değildir.

UYARI:
Bu çıktı yalnızca semptom temelli karar destek amaçlıdır. Tıbbi tanı değildir.

BENZER VAKALAR (TR):


,text_tr,label_tr,similarity
0,boğaz ağrısı; öksürük; ateş,akut sinüzit,0.779195
1,boğaz ağrısı; öksürük; ateş,kronik sinüzit,0.779195
2,boğaz ağrısı; ateş,kronik obstrüktif akciğer hastalığı (koah),0.755435
3,boğaz ağrısı; ateş,akut sinüzit,0.755435
4,boğaz ağrısı; hırıltı; ateş,kronik obstrüktif akciğer hastalığı (koah),0.722367


In [61]:
print([name for name in globals() if "predict" in name])

['get_top_k_predictions', 'predict_with_retrieval', 'full_predict', 'translate_top_predictions']


In [62]:
def run_user_text_direct(user_text):
    # 1) kullanıcı metnini İngilizceye çevir
    translated_input = translate_to_en(user_text)

    # 2) modeli doğrudan bu metinle çalıştır
    result_en = predict_with_retrieval(
        user_text=translated_input,
        baseline_model=baseline_model,
        classes=classes,
        retrieval_df=retrieval_df,
        retrieval_embeddings=retrieval_embeddings,
        embed_model=embed_model,
        top_k_cls=3,
        top_k_ret=5
    )

    # 3) gösterim için alanları tamamla
    result_en["summary"] = translated_input
    result_en["pattern_note"] = "Retrieved cases suggest a symptom similarity pattern."
    result_en["final_comment"] = (
        "These outputs represent possible conditions based on the user's complaint "
        "and similarity matching. They are not a medical diagnosis."
    )

    if "retrieved_label_summary" not in result_en:
        result_en["retrieved_label_summary"] = result_en["retrieved_cases"]["label"].value_counts().to_dict()

    # 4) Türkçeleştir
    result_tr = localize_result_to_tr(result_en)

    # 5) çıktı ver
    print("KULLANICI HAM GİRİŞİ:")
    print(user_text)

    print("\nMODELİN GÖRDÜĞÜ İNGİLİZCE METİN:")
    print(translated_input)

    print("\n--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---")
    print_localized_result_tr(result_tr)

    return result_en, result_tr

In [63]:
long_text = """
Dün arkadaşlarla pikniğe gittik. Akşam eve dönünce çok halsiz hissettim.
Gece boğazım yanmaya başladı. Sabah kalkınca öksürüğüm vardı, biraz ateşim çıkmış gibi hissettim.
Başım da ağrıyor. Çok ciddi mi bilmiyorum ama son iki gündür arttı gibi.
"""

result_en, result_tr = run_user_text_direct(long_text)

KULLANICI HAM GİRİŞİ:

Dün arkadaşlarla pikniğe gittik. Akşam eve dönünce çok halsiz hissettim.
Gece boğazım yanmaya başladı. Sabah kalkınca öksürüğüm vardı, biraz ateşim çıkmış gibi hissettim.
Başım da ağrıyor. Çok ciddi mi bilmiyorum ama son iki gündür arttı gibi.


MODELİN GÖRDÜĞÜ İNGİLİZCE METİN:
We went on a picnic with friends yesterday. When I returned home in the evening, I felt very weak.
My throat started burning at night. When I woke up in the morning, I had a cough and felt like I had a bit of a fever.
My head hurts too. I don't know if it's serious, but it seems to have increased in the last two days.

--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---
KULLANICI GİRİŞİ:
Dün arkadaşlarla pikniğe gittik. Akşam eve döndüğümde kendimi çok halsiz hissettim.
Geceleri boğazım yanmaya başladı. Sabah uyandığımda öksürüğüm vardı ve biraz ateşim varmış gibi hissettim.
Benim de başım ağrıyor. Ciddi mi bilmiyorum ama son iki günde artmış gibi görünüyor.

ÖZET:
Dün arkadaşlarla pikniğe gittik.

,text_tr,label_tr,similarity
0,boğaz ağrısı; ateş,akut sinüzit,0.696372
1,boğaz ağrısı; ateş,kronik obstrüktif akciğer hastalığı (koah),0.696372
2,boğaz ağrısı; öksürük; ateş,akut sinüzit,0.693239
3,boğaz ağrısı; öksürük; ateş,kronik sinüzit,0.693239
4,boğaz ağrısı; ön baş ağrısı; ateş,kronik sinüzit,0.656201


In [64]:
result_en, result_tr = run_full_system_tr("Öksürüğüm, boğaz ağrım ve ateşim var")

MODELİN GÖRDÜĞÜ İNGİLİZCE INPUT:
I have a cough, sore throat and fever

--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---
KULLANICI GİRİŞİ:
Öksürüğüm, boğaz ağrım ve ateşim var

ÖZET:
Hastanın bildirdiği şikayet: Öksürüğüm, boğaz ağrım ve ateşim var

OLASI DURUMLAR:
1. alerji -> 0.2442
2. gastroözofageal reflü hastalığı -> 0.0713
3. nezle, soğuk algınlığı -> 0.0659

ÖRÜNTÜ NOTU:
Alınan vakalar bir semptom benzerliği modelini akla getiriyor.

SON YORUM:
Bu çıktılar semptom benzerliğine dayalı olası koşulları temsil eder ve tıbbi bir teşhis değildir.

UYARI:
Bu çıktı yalnızca semptom temelli karar destek amaçlıdır. Tıbbi tanı değildir.

BENZER VAKALAR (TR):


,text_tr,label_tr,similarity
0,boğaz ağrısı; öksürük; ateş,akut sinüzit,0.847214
1,boğaz ağrısı; öksürük; ateş,kronik sinüzit,0.847214
2,boğaz ağrısı; ateş,akut sinüzit,0.786924
3,boğaz ağrısı; ateş,kronik obstrüktif akciğer hastalığı (koah),0.786924
4,boğaz ağrısı; hırıltı; ateş,kronik obstrüktif akciğer hastalığı (koah),0.783300


In [65]:
result_en, result_tr = run_full_system_tr("İdrar yaparken yanma ve ateşim var")

MODELİN GÖRDÜĞÜ İNGİLİZCE INPUT:
I have fever and burning sensation while urinating.

--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---
KULLANICI GİRİŞİ:
İdrar yaparken ateş ve yanma hissi var.

ÖZET:
Hastanın bildirdiği şikayet: İdrar yaparken ateşim ve yanma hissim var.

OLASI DURUMLAR:
1. peptik ülser hastalığı -> 0.0892
2. idrar yolu enfeksiyonu -> 0.0712
3. impetigo -> 0.0712

ÖRÜNTÜ NOTU:
Alınan vakalar bir semptom benzerliği modelini akla getiriyor.

SON YORUM:
Bu çıktılar semptom benzerliğine dayalı olası koşulları temsil eder ve tıbbi bir teşhis değildir.

UYARI:
Bu çıktı yalnızca semptom temelli karar destek amaçlıdır. Tıbbi tanı değildir.

BENZER VAKALAR (TR):


,text_tr,label_tr,similarity
0,ağrılı idrara çıkma,endoftalmi,0.670087
1,ağrılı idrara çıkma,vajinada yabancı cisim,0.670087
2,ağrılı idrara çıkma,vajinada yabancı cisim,0.670087
3,bulantı; ağrılı idrara çıkma,lomber ponksiyon sonrası baş ağrısı,0.667298
4,bulantı; ağrılı idrara çıkma,lomber ponksiyon sonrası baş ağrısı,0.667298


In [66]:
result_en, result_tr = run_full_system_tr("Cildimde kaşıntı ve kızarıklık var")

MODELİN GÖRDÜĞÜ İNGİLİZCE INPUT:
I have itching and redness on my skin

--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---
KULLANICI GİRİŞİ:
Cildimde kaşıntı ve kızarıklık var

ÖZET:
Hastanın bildirdiği şikayet: Cildimde kaşıntı ve kızarıklık var

OLASI DURUMLAR:
1. sedef hastalığı -> 0.1490
2. ilaç reaksiyonu -> 0.0772
3. mantar enfeksiyonu -> 0.0765

ÖRÜNTÜ NOTU:
Alınan vakalar bir semptom benzerliği modelini akla getiriyor.

SON YORUM:
Bu çıktılar semptom benzerliğine dayalı olası koşulları temsil eder ve tıbbi bir teşhis değildir.

UYARI:
Bu çıktı yalnızca semptom temelli karar destek amaçlıdır. Tıbbi tanı değildir.

BENZER VAKALAR (TR):


,text_tr,label_tr,similarity
0,cildin kaşınması; cilt tahrişi; deri döküntüsü,aktinik keratoz,0.740768
1,anormal görünen cilt; ciltte kaşıntı,aktinik keratoz,0.687835
2,anormal görünen cilt; yüz belirtileri; ciltte ...,aktinik keratoz,0.685522
3,vajinal kaşıntı; vajinal kızarıklık,vajinal mantar enfeksiyonu,0.678877
4,hasta hissetmek; cildin kaşınması; deri döküntüsü,alopesi,0.674769


In [68]:
result_en, result_tr = run_full_system_tr("Mide bulantım ve karın ağrım var")

MODELİN GÖRDÜĞÜ İNGİLİZCE INPUT:
I have nausea and abdominal pain

--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---
KULLANICI GİRİŞİ:
Mide bulantım ve karın ağrım var

ÖZET:
Hastanın bildirdiği şikayet: Mide bulantısı ve karın ağrım var

OLASI DURUMLAR:
1. tifo -> 0.1048
2. peptik ülser hastalığı -> 0.0747
3. ilaç reaksiyonu -> 0.0724

ÖRÜNTÜ NOTU:
Alınan vakalar bir semptom benzerliği modelini akla getiriyor.

SON YORUM:
Bu çıktılar semptom benzerliğine dayalı olası koşulları temsil eder ve tıbbi bir teşhis değildir.

UYARI:
Bu çıktı yalnızca semptom temelli karar destek amaçlıdır. Tıbbi tanı değildir.

BENZER VAKALAR (TR):


,text_tr,label_tr,similarity
0,bulantı; üst karın ağrısı,akut pankreatit,0.812728
1,bulantı; ishal; üst karın ağrısı,akut pankreatit,0.776596
2,bulantı; alt karın ağrısı; sırt ağrısı,pyelonefrit,0.762546
3,bulantı; sırt ağrısı; üst karın ağrısı,akut pankreatit,0.760619
4,bulantı; alt vücut ağrısı; üst karın ağrısı,akut pankreatit,0.759788


In [69]:
result_en, result_tr = run_full_system_tr("Nefes darlığım ve göğsümde sıkışma var")

MODELİN GÖRDÜĞÜ İNGİLİZCE INPUT:
I have shortness of breath and tightness in my chest

--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---
KULLANICI GİRİŞİ:
Nefes darlığı ve göğüs kafesimde sıkışma var

ÖZET:
Hastanın bildirdiği şikayet: Nefes darlığı ve göğüs kafesimde sıkışma var

OLASI DURUMLAR:
1. gastroözofageal reflü hastalığı -> 0.1525
2. alerji -> 0.0777
3. ilaç reaksiyonu -> 0.0729

ÖRÜNTÜ NOTU:
Alınan vakalar bir semptom benzerliği modelini akla getiriyor.

SON YORUM:
Bu çıktılar semptom benzerliğine dayalı olası koşulları temsil eder ve tıbbi bir teşhis değildir.

UYARI:
Bu çıktı yalnızca semptom temelli karar destek amaçlıdır. Tıbbi tanı değildir.

BENZER VAKALAR (TR):


,text_tr,label_tr,similarity
0,nefes darlığı; göğüs sıkışması,panik atak,0.883268
1,nefes darlığı; göğüs sıkışması,astım,0.883268
2,nefes darlığı; göğüs sıkışması,torasik aort anevrizması,0.883268
3,nefes darlığı; göğüs sıkışması,torasik aort anevrizması,0.883268
4,nefes darlığı; göğüs sıkışması,koroner ateroskleroz,0.883268


In [70]:
import joblib
import numpy as np

model_dir = "/content/drive/MyDrive/medical_triage_project/models/"

joblib.dump(baseline_model, model_dir + "classifier.pkl")
np.save(model_dir + "retrieval_embeddings.npy", retrieval_embeddings)
retrieval_df.to_csv(model_dir + "retrieval_data.csv", index=False)

In [72]:
# =========================
# FINAL DEMO
# =========================

user_input = input("Şikayetinizi yazın: ")

result_en, result_tr = run_full_system_tr(user_input)

Şikayetinizi yazın: Nefes darlığım ve göğsümde sıkışma var
MODELİN GÖRDÜĞÜ İNGİLİZCE INPUT:
I have shortness of breath and tightness in my chest

--- KULLANICIYA GÖSTERİLEN TÜRKÇE ÇIKTI ---
KULLANICI GİRİŞİ:
Nefes darlığı ve göğüs kafesimde sıkışma var

ÖZET:
Hastanın bildirdiği şikayet: Nefes darlığı ve göğüs kafesimde sıkışma var

OLASI DURUMLAR:
1. gastroözofageal reflü hastalığı -> 0.1525
2. alerji -> 0.0777
3. ilaç reaksiyonu -> 0.0729

ÖRÜNTÜ NOTU:
Alınan vakalar bir semptom benzerliği modelini akla getiriyor.

SON YORUM:
Bu çıktılar semptom benzerliğine dayalı olası koşulları temsil eder ve tıbbi bir teşhis değildir.

UYARI:
Bu çıktı yalnızca semptom temelli karar destek amaçlıdır. Tıbbi tanı değildir.

BENZER VAKALAR (TR):


,text_tr,label_tr,similarity
0,nefes darlığı; göğüs sıkışması,panik atak,0.883268
1,nefes darlığı; göğüs sıkışması,astım,0.883268
2,nefes darlığı; göğüs sıkışması,torasik aort anevrizması,0.883268
3,nefes darlığı; göğüs sıkışması,torasik aort anevrizması,0.883268
4,nefes darlığı; göğüs sıkışması,koroner ateroskleroz,0.883268
